In [1]:
import os
import csv
from Bio import AlignIO

def calculate_identity(seq1, seq2):
    """
    Calcula el porcentaje de identidad entre dos secuencias alineadas,
    excluyendo posiciones con gaps en cualquiera de las dos.
    """
    matches = sum(a == b for a, b in zip(seq1, seq2) if a != '-' and b != '-')
    length = sum((a != '-' and b != '-') for a, b in zip(seq1, seq2))
    return (matches / length) * 100 if length > 0 else 0

def is_frame_hit(hit_id):
    """
    Determina si un ID corresponde a un marco de lectura (frame/orf).
    Puedes ajustar los criterios si tus IDs usan otro formato.
    """
    hit_id_lower = hit_id.lower()
    return "frame" in hit_id_lower or "orf" in hit_id_lower

def process_alignment_file(file_path, output_folder):
    """
    Procesa un archivo .aln comparando la secuencia query (primera) contra cada hit.
    Guarda los % de identidad en un CSV individual y retorna info para el resumen.
    """
    alignment = AlignIO.read(file_path, "clustal")
    query_id = alignment[0].id
    query_seq = alignment[0].seq

    results = []
    identity_values = []

    for i in range(1, len(alignment)):
        hit_id = alignment[i].id
        hit_seq = alignment[i].seq
        identity = calculate_identity(query_seq, hit_seq)
        results.append([query_id, hit_id, f"{identity:.2f}"])

        # Solo incluir si no es un frame
        if not is_frame_hit(hit_id):
            identity_values.append(identity)

    # Guardar CSV individual
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    output_file = os.path.join(output_folder, f"{base_name}_identity.csv")

    with open(output_file, "w", newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Query", "Hit", "% Identidad"])
        writer.writerows(results)

    print(f"Guardado: {output_file}")

    # Retornar info para resumen general (solo promedios sin frames)
    return [
        base_name + ".aln",
        query_id,
        len(identity_values),
        f"{sum(identity_values)/len(identity_values):.2f}" if identity_values else "0.00"
    ]

def process_all_alignments(input_folder, output_folder):
    """
    Procesa todos los archivos .aln y genera archivos CSV individuales
    más un resumen general con promedios de identidad.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    resumen_general = []

    for filename in os.listdir(input_folder):
        if filename.endswith(".aln"):
            file_path = os.path.join(input_folder, filename)
            resumen = process_alignment_file(file_path, output_folder)
            resumen_general.append(resumen)

    # Guardar archivo de resumen general
    resumen_file = os.path.join(output_folder, "resumen_identidad_general.csv")
    with open(resumen_file, "w", newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Archivo", "Query_ID", "Num_Hits_Validos", "Promedio_%_Identidad"])
        writer.writerows(resumen_general)

    print(f"\nResumen general guardado en: {resumen_file}")

# === Configuración de rutas ===
input_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\alineamiento_multiple"
output_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\porcentaje_identidad"

if __name__ == "__main__":
    print("=== Calculando % de identidad (Query vs Hit) ===")
    process_all_alignments(input_folder, output_folder)
    print("\n✅ Proceso finalizado.")


=== Calculando % de identidad (Query vs Hit) ===
Guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\porcentaje_identidad\Aligned_1_identity.csv
Guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\porcentaje_identidad\Aligned_10_identity.csv
Guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\porcentaje_identidad\Aligned_11_identity.csv
Guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_